# 🎨 Fooocus Designer 2.0 — Turnkey Graphic Design Appliance
### Professional Commercial Graphic Asset Production on Google Colab (Free T4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiL4gh/Fooocus-Design-Tool/blob/main/Fooocus_Designer_2_0.ipynb)

---

### ⚡ What is Fooocus Designer 2.0?
A zero-friction graphic design tool powered by **RunDiffusion/Juggernaut-XL-v9** and **ByteDance SDXL-Lightning**.

- 🎯 **Zero Technical Slider Tweaking:** Select a category (*Silhouette, Flat Vector, Sticker, Logo, Pattern, Poster*) and type your prompt.
- ⚡ **Dual-Speed Engine:** **⚡ Fast (~3s)** for rapid iteration vs **🎯 Master (~15s)** for 100% LoRA detail.
- 🏷️ **Baked Category LoRAs:** Category-specific LoRAs pre-configured with trigger words and microstock negative prompts.
- 🤖 **Customizable Model Architecture:** Premade with Juggernaut XL v9; 1-click selectable base models (Animagine, RealVisXL, SDXL Base).
- 🎨 **Fooocus Style Engine:** Official Fooocus & SAI style templates with one-click multi-selection.
- 📥 **Drop-to-Load Metadata:** Drop any previously generated PNG back into the tool to instantly restore all settings.
- 🎨 **Curated Color Palettes:** 8 commercial design palettes (*Cyberpunk, Pastel, Boho, Luxury Gold, etc.*) auto-populate color swatches.
- 🧹 **Automatic Transparency & Vectorization:** Instant background removal via `rembg` and optional SVG conversion via `StarVector`.

> **⚠️ Important GPU Requirement:** Ensure you have GPU acceleration enabled:
> **Runtime** ➔ **Change runtime type** ➔ **Hardware accelerator** ➔ **T4 GPU**.

In [ ]:
#@title 1. Verify GPU Acceleration
import subprocess
try:
    gpu_info = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode('utf-8').strip()
    print(f"✅ GPU Detected: {gpu_info}")
except Exception:
    print("❌ No GPU detected! Please go to Runtime -> Change runtime type -> Select T4 GPU and reconnect.")

In [ ]:
#@title 2. Launch Fooocus Designer 2.0 (1-Click Setup & Launch)
#@markdown Select your preferred setup options:
Preload_Models = False #@param {type:"boolean"}
Use_Google_Drive_For_Outputs = True #@param {type:"boolean"}

import os, sys

# 1. Mount Google Drive if requested
output_dir_symlink = None
if Use_Google_Drive_For_Outputs:
    try:
        from google.colab import drive
        if not os.path.exists('/content/drive/MyDrive'):
            print("📁 Mounting Google Drive to preserve generated assets...")
            drive.mount('/content/drive')
        else:
            print("📁 Google Drive is already mounted.")
        drive_out = '/content/drive/MyDrive/Fooocus_Designer_Outputs'
        os.makedirs(drive_out, exist_ok=True)
        output_dir_symlink = drive_out
        print(f"✅ Generated assets will be saved directly to: {drive_out}")
    except Exception as e:
        print(f"⚠️ Could not mount Google Drive: {e}. Saving to local session instead.")

# 2. Anchor to /content and clean up any accidental nested folder
%cd /content
if os.path.exists('/content/Fooocus-Design-Tool/Fooocus-Design-Tool'):
    print("🧹 Cleaning up nested directory from previous run...")
    !rm -rf /content/Fooocus-Design-Tool/Fooocus-Design-Tool

if not os.path.exists('/content/Fooocus-Design-Tool/launch.py'):
    print("\n📦 Cloning Fooocus-Design-Tool...")
    !git clone https://github.com/NiL4gh/Fooocus-Design-Tool.git
else:
    print("\n📦 Updating Fooocus-Design-Tool to latest version...")
    %cd /content/Fooocus-Design-Tool
    !git reset --hard
    !git pull
    %cd /content

%cd /content/Fooocus-Design-Tool

# 3. Link outputs to Google Drive if mounted
if output_dir_symlink:
    if os.path.exists('outputs') and not os.path.islink('outputs'):
        !rm -rf outputs
    if not os.path.exists('outputs'):
        try:
            os.symlink(output_dir_symlink, 'outputs')
            print("🔗 Outputs symlinked to Google Drive.")
        except Exception:
            pass

# 4. Remove incompatible torchao if present on Colab
!pip uninstall -y torchao -q

# 5. Install dependencies with live real-time output
print("\n📥 Installing dependencies (takes ~1-2 minutes on first run)...")
!pip install -r requirements.txt

# 6. Pre-download SDXL weights with live progress bars
if Preload_Models:
    print("\n" + "=" * 60)
    print("📦 Pre-downloading Base Model: RunDiffusion/Juggernaut-XL-v9 (SDXL FP16 ~6.6GB)")
    print("⚡ Pre-downloading Speed Adapter: ByteDance/SDXL-Lightning (4-step LoRA ~390MB)")
    print("🏷️ Categories: Baked commercial design LoRAs (zero-reload dynamic routing)")
    print("=" * 60)
    print("   Showing real-time download progress:")
    !python -u -c "from modules.sdxl_pipeline import load_pipeline, unload_pipeline; load_pipeline(speed_mode='fast'); unload_pipeline(); print('\n✅ SDXL models and speed adapters cached successfully!')"

# 7. Launch web UI with public share link
print("\n🚀 Launching Fooocus Designer 2.0...")
print("=" * 60)
!python -u launch.py --share

In [ ]:
#@title 3. (Optional) Setup ngrok Tunnel for Rock-Solid Persistent URL
#@markdown If the standard Gradio live link disconnects during long sessions, use an ngrok token from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok_token = "" #@param {type:"string"}

if ngrok_token.strip():
    %cd /content/Fooocus-Design-Tool
    !pip install pyngrok
    from pyngrok import ngrok
    ngrok.set_auth_token(ngrok_token.strip())
    tunnel = ngrok.connect(7865)
    print(f"\n🌐 Public ngrok URL: {tunnel.public_url}")
    print("🚀 Launching Fooocus Designer 2.0...")
    !python -u launch.py --no-share
else:
    print("ℹ️ Paste your ngrok auth token above if you wish to use an ngrok tunnel.")